In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
cars_df = pd.read_csv('../datasets/raw_data/cars_info.csv')

In [3]:
df = cars_df.copy()
df.head()

,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191,NaN,NaN
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214,NaN,NaN
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140,NaN,NaN
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255,NaN,NaN
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244,NaN,NaN


In [4]:
df.shape

(10060, 15)

In [5]:
df.columns

Index(['Model year', 'Make', 'Model', 'Vehicle class', 'Engine size (L)',
       'Cylinders', 'Transmission', 'Fuel type', 'City (L/100 km)',
       'Highway (L/100 km)', 'Combined (L/100 km)', 'Combined (mpg)',
       'CO2 emissions (g/km)', 'CO2 rating', 'Smog rating'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10060 entries, 0 to 10059
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model year            10060 non-null  int64  
 1   Make                  10060 non-null  object 
 2   Model                 10060 non-null  object 
 3   Vehicle class         10060 non-null  object 
 4   Engine size (L)       10060 non-null  float64
 5   Cylinders             10060 non-null  int64  
 6   Transmission          10060 non-null  object 
 7   Fuel type             10060 non-null  object 
 8   City (L/100 km)       10060 non-null  float64
 9   Highway (L/100 km)    10060 non-null  float64
 10  Combined (L/100 km)   10060 non-null  float64
 11  Combined (mpg)        10060 non-null  int64  
 12  CO2 emissions (g/km)  10060 non-null  int64  
 13  CO2 rating            8932 non-null   float64
 14  Smog rating           7826 non-null   float64
dtypes: float64(6), int6

##### Duplicate rows check

In [ ]:
df.duplicated().sum()

np.int64(0)

##### Discarding all the irrelavant columns

1. Discarding Combined(mpg) column

In [7]:
df = df.drop(columns = 'Combined (mpg)')

2. Verifying whether Combined(L/100km) values are accuarte or not before dropping.

In [8]:
expected_values_of_combined = round((df['City (L/100 km)'] * 0.55) + (df['Highway (L/100 km)'] * 0.45),1)
expected_values_of_combined

0         8.4
1         9.3
2         6.0
3        11.1
4        10.6
         ... 
10055     8.9
10056     9.5
10057     9.0
10058     9.6
10059     9.9
Length: 10060, dtype: float64

In [ ]:
temp_df = pd.DataFrame(
    {
        'Combined':df['Combined (L/100 km)'],
        'Expected': expected_values_of_combined
    }
)
temp_df

,Combined,Expected
0,8.3,8.4
1,9.3,9.3
2,6.1,6.0
3,11.1,11.1
4,10.6,10.6
...,...,...
10055,8.9,8.9
10056,9.5,9.5
10057,9.0,9.0
10058,9.6,9.6


In [10]:
matches = temp_df[temp_df['Combined'] == temp_df['Expected']].shape[0]
print('Total no of matched rows =', matches)

unmatched = temp_df[temp_df['Combined'] != temp_df['Expected']].shape[0]
print('Total no of unmatched rows =', unmatched)

Total no of matched rows = 8313
Total no of unmatched rows = 1747


In [11]:
temp_df['Error'] = round(abs(temp_df['Combined'] - temp_df['Expected']), 1)
temp_df['Error'].unique()

array([0.1, 0. , 0.2, 0.4, 0.5, 0.3])

In [12]:
temp_df['Error'].value_counts()

Error
0.0    8313
0.1    1686
0.2      33
0.4      13
0.3      11
0.5       4
Name: count, dtype: int64

In [13]:
temp_df['Signed_error'] = (temp_df['Combined'] - temp_df['Expected']).round(1)
print(temp_df['Signed_error'].value_counts().sort_index())

Signed_error
-0.4      10
-0.3       9
-0.2      23
-0.1     885
 0.0    8313
 0.1     801
 0.2      10
 0.3       2
 0.4       3
 0.5       4
Name: count, dtype: int64


Conclusion: Absolute error between Combined(L/100km) and expected_values_of_combined = [0.55 * City(L/100km)] + [0.45 * Highway(L/100km)] were rounding artefacts not data entry errors. Hence Combined(L/100km) is accurate and its safe to drop City(L/100km) and Highway(L/100km).

3. Discarding City and Highway column

In [14]:
df = df.drop(columns = ['City (L/100 km)', 'Highway (L/100 km)'])

4. Discarding Smog rating column

In [15]:
df = df.drop(columns = 'Smog rating')

5. Discarding CO2 rating column

In [16]:
df = df.drop(columns = 'CO2 rating')

6. Tranferring the cleaned table to cleaned_data folder

In [17]:
df.to_csv('../datasets/cleaned_data/cleaned_cars_info.csv', index = False)